In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pandas import read_csv
import seaborn as sns
import numpy as np
import sys
import os
os.makedirs("../artifacts", exist_ok=True)
sys.path.append("..")
from src.components.data_ingestion import DataLoader,engine
from src.pipeline.main import KpiBy_hotel,Kpi_to_file_json
from src.components.kpi_engine import HotelKPIEngine


loader = DataLoader("cleaned_hotel_data",engine)
df = loader.load()


# I split the data and placed into artifacts folder for ML model.

Resort_hotel = df[df['hotel'] == 'Resort Hotel']
Resort_hotel.to_csv("../artifacts/Resort_hotel.csv", index=False)

City_hotel = df[df['hotel'] == 'City Hotel']
City_hotel.to_csv("../artifacts/City_hotel.csv", index=False)




[ 2026-04-24 21:47:18,562 ] — Line: 94 — hotel_pricing — INFO — DataLoader initialized for table: cleaned_hotel_data
[ 2026-04-24 21:47:18,562 ] — Line: 102 — hotel_pricing — INFO — Loading data from table: cleaned_hotel_data
[ 2026-04-24 21:47:19,194 ] — Line: 105 — hotel_pricing — INFO — Data loaded successfully. Shape: (55713, 22)
[ 2026-04-24 21:47:19,478 ] — Line: 42 — hotel_pricing — INFO — Hotel data successfully split into CSV files.
[ 2026-04-24 21:47:23,750 ] — Line: 54 — hotel_pricing — INFO — Hotel data successfully uploaded into SQL tables.
[ 2026-04-24 21:47:23,821 ] — Line: 50 — hotel_pricing — INFO — Initializing KPI Engine for: City_hotel
[ 2026-04-24 21:47:23,825 ] — Line: 57 — hotel_pricing — INFO — Filtering data for year: 2016
[ 2026-04-24 21:47:23,876 ] — Line: 94 — hotel_pricing — INFO — DataLoader initialized for table: cleaned_hotel_data
[ 2026-04-24 21:47:23,877 ] — Line: 102 — hotel_pricing — INFO — Loading data from table: cleaned_hotel_data
[ 2026-04-24 21:

In [3]:
# Exploratory Data Analysis for the whole data frame

df.head(5)
df.describe()
df.isnull().sum()
df.columns
df.dtypes
df.shape
df['adr'].describe()
df['adr'].nlargest(10)




55623    451.5
5937     384.0
5976     382.0
55524    375.5
3640     369.0
18029    367.0
45935    365.0
13398    359.0
13639    359.0
13523    357.0
Name: adr, dtype: float64

In [4]:
# # -----------------------------
# # 1. Drop assigned_room_type
# # -----------------------------
# # They are duplicates; reserved_room_type is the correct one
df.drop(['assigned_room_type'], axis=1, inplace=True)


# # -----------------------------
# # 2. Rename reserved_room_type → room_class
# # -----------------------------
df.rename(columns={'reserved_room_type': 'room_class'}, inplace=True)


# # -----------------------------
# # 3. Remove ADR outliers (<5 or >1000)
# # -----------------------------
df = df[(df['adr'] > 5) & (df['adr'] < 1000)]


# # -----------------------------
# # 4. Remove Complementary segment
# # -----------------------------
df = df[df['market_segment'] != 'Complementary']


# # -----------------------------
# # 5. Remove Undefined distribution channel
# # -----------------------------
df = df[df['distribution_channel'] != 'Undefined']


# # -----------------------------
# # 6. Remove room type P (0 ADR)
# # -----------------------------
df = df[df['room_class'] != 'P']


# # -----------------------------
# # 7. Map room_class codes → descriptive names
# # -----------------------------
room_map = {
    'A': 'Standard King',
    'B': 'Standard Twin',
    'C': 'Premier King',
    'D': 'Premier Twin',
    'E': 'Club King',
    'F': 'Club Twin',
    'G': 'Suite King',
    'H': 'Executive Suite',
    'I': 'Deluxe King',
    'K': 'Deluxe Twin',
    'L': 'Presidential Suite'
}

df['room_class'] = df['room_class'].replace(room_map)


# -----------------------------
# 8. Merge children + babies → Kids
# -----------------------------
df['Kids'] = df['children'].astype(int) + df['babies'].astype(int)


# -----------------------------
# 9. Drop original children/babies
# -----------------------------
df.drop(['children', 'babies'], axis=1, inplace=True)

# -----------------------------
# 10. Drop original children/babies
# -----------------------------

df = df[~df['arrival_date_year'].isin([2015, 2017])]




KeyError: "['assigned_room_type'] not found in axis"

In [ ]:
''' I checked the yearly growth and found there's incomplete data in 2015, and 2017 so I will remove these years.'''

for h in df['hotel'].unique():
    
    temp = df[df['hotel'] == h]

    # Chronological order for consistent plotting
    month_order = [
        'January', 'February', 'March', 'April', 'May', 'June', 
        'July', 'August', 'September', 'October', 'November', 'December'
    ]

    data_pivot_year = temp.pivot_table(
        index='arrival_date_year', 
        columns='arrival_date_month', 
        values='adr', 
        aggfunc='mean'
    ).reindex(columns=month_order)


    plt.figure(figsize=(14, 6))
    sns.heatmap(data_pivot_year, cmap='YlGnBu', annot=True, fmt=".1f", linewidths=.5)
    plt.title(f'{h}: Yield Growth (Average ADR by Year/Month)', fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()